# 🆘 SafeSOS — Train MobileNetV3 on Colab

This notebook trains the deep-learning model on a Colab T4/L4 GPU.

## Prerequisites (do these once on your Mac before running this notebook)

1. **Push your SafeSOS repo to GitHub**
   ```bash
   git push origin <your-branch>
   ```

2. **Tarball the processed data and upload to Drive**
   ```bash
   cd SafeSOS
   tar czf data_processed.tar.gz data/processed
   # ~600MB; upload to Google Drive at:
   #   MyDrive/SafeSOS-data/data_processed.tar.gz
   ```

3. **Make sure Colab runtime has a GPU**
   Runtime → Change runtime type → Hardware accelerator → **T4 GPU** (or L4 if available)

## What this notebook does

1. Mounts Drive
2. Clones your GitHub repo
3. Extracts the data
4. Installs deps
5. Trains MobileNetV3-Small + fits temperature scaling
6. Writes the trained model back to Drive


## 1. Configuration

Edit these two variables to match your GitHub username and branch.

In [ ]:
# 👇 EDIT THESE
GITHUB_USER = ""   # your GitHub username
BRANCH      = ""      # the branch containing the latest code

# Where you uploaded the data tarball on Drive
DATA_TAR_PATH = ""

# Where to copy trained checkpoints back to
CHECKPOINT_DRIVE_DIR = ""

# Training hyperparams (sensible defaults for T4)
EPOCHS = 12
BATCH_SIZE = 128
LR = 3e-4
EARLY_STOP_PATIENCE = 4


## 2. Mount Google Drive

Sign in with the account that has the data tarball.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Sanity check: confirm the data tarball exists
import os
assert os.path.isfile(DATA_TAR_PATH), \
    f"Data tarball not found at {DATA_TAR_PATH}. Upload it first."
size_mb = os.path.getsize(DATA_TAR_PATH) / (1024 * 1024)
print(f"✓ Found data tarball: {DATA_TAR_PATH} ({size_mb:.0f} MB)")


## 3. GPU sanity check

Confirms a CUDA GPU is available. If not, go to Runtime → Change runtime type.

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), \
    "No CUDA GPU detected. Change runtime: Runtime → Change runtime type → GPU"
print(f"✓ CUDA device: {torch.cuda.get_device_name(0)}")
print(f"✓ torch       : {torch.__version__}")


## 4. Clone the SafeSOS repo

We clone fresh each run so the notebook always picks up your latest commits.

In [ ]:
import os, shutil

# Clean previous clone (if any)
if os.path.isdir("/content/SafeSOS"):
    shutil.rmtree("/content/SafeSOS")

!cd /content && git clone -b "{BRANCH}" "https://github.com/{GITHUB_USER}/SafeSOS.git"
assert os.path.isdir("/content/SafeSOS"), "Clone failed — check GITHUB_USER and BRANCH above"

%cd /content/SafeSOS
!git log -1 --oneline


## 5. Extract the data tarball

Should produce `data/processed/{train,val,test}/<class>/*.jpg`.

In [ ]:
!tar xzf "{DATA_TAR_PATH}" -C .

# Verify structure
import os
expected = ["data/processed/train", "data/processed/val", "data/processed/test"]
for p in expected:
    assert os.path.isdir(p), f"Missing {p}"
    classes = sorted(os.listdir(p))
    print(f"{p}: {len(classes)} classes -> {classes[:5]}...")

!du -sh data/processed/train data/processed/val data/processed/test


In [ ]:
# Clean macOS metadata files that snuck into the tarball
!find data/processed -name "._*" -delete
!find data/processed -name ".DS_Store" -delete

# Verify
!find data/processed -name "._*" | wc -l   # should print 0
!ls data/processed/train/call/ | head -5     # should show real .jpg files

## 6. Install dependencies

Colab already has torch/torchvision. We just need to make sure the project's
imports work.

In [ ]:
# Colab usually has these but pin versions just in case
!pip install -q "torch>=2.0" "torchvision>=0.15" "tqdm>=4.65" "pillow>=10.0"

# Smoke-test that the project's modules import
import sys
sys.path.insert(0, "/content/SafeSOS")
from models.deep_learning import SelectiveMobileNet
from models.naive_baseline import MajorityClassifier
print("✓ project modules import cleanly")


## 7. Train MobileNetV3-Small

This is the main event. Watch the val_acc column — it should climb from
~30% at epoch 1 to ~95-99% by epoch 8-10. If it plateaus early, that's
fine — early stopping kicks in after `EARLY_STOP_PATIENCE` non-improving
epochs.

**Expected wall time on T4:** ~2-3 minutes per epoch × 12 = 25-35 minutes total.

In [ ]:
!python scripts/train_dl.py \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --lr {LR} \
    --num-workers 2 \
    --early-stop-patience {EARLY_STOP_PATIENCE}


## 8. Inspect training outputs

Quick look at the metadata file (config, temperature, test accuracy, history).

In [ ]:
import json
with open("models/checkpoints/mobilenet_meta.json") as f:
    meta = json.load(f)

print(f"Classes ({len(meta['class_names'])}):")
for i, c in enumerate(meta['class_names']):
    print(f"  {i}: {c}")

print(f"\nBest val accuracy : {meta['best_val_acc']:.4f}")
print(f"Test accuracy     : {meta['test_acc']:.4f}")
print(f"Temperature (T)   : {meta['temperature']:.4f}")
print(f"Epochs trained    : {len(meta['history'])}")

print("\nPer-epoch history:")
print(f"  {'epoch':>5s} {'train_loss':>12s} {'val_loss':>10s} {'val_acc':>10s}")
for h in meta['history']:
    print(f"  {h['epoch']:>5d} {h['train_loss']:>12.4f} {h['val_loss']:>10.4f} {h['val_acc']:>10.4f}")


## 9. Copy trained artifacts back to Drive

So you can download them to your Mac for the Gradio app + report.

In [ ]:
import os, shutil

os.makedirs(CHECKPOINT_DRIVE_DIR, exist_ok=True)

for fname in ["mobilenet_best.pth", "mobilenet_meta.json"]:
    src = f"models/checkpoints/{fname}"
    dst = f"{CHECKPOINT_DRIVE_DIR}/{fname}"
    shutil.copy2(src, dst)
    size_mb = os.path.getsize(dst) / (1024 * 1024)
    print(f"✓ {src} -> {dst} ({size_mb:.1f} MB)")

print(f"\nAll set. Download from Drive to your Mac:")
print(f"  {CHECKPOINT_DRIVE_DIR}/")


## 10. Sanity-check inference on a few test images

Loads the saved checkpoint with selective prediction and runs it on 5 random
test images to confirm everything is wired up. Useful before you bother
downloading the checkpoint.

In [ ]:
import random
from pathlib import Path
import torch
from PIL import Image
from torchvision import transforms
from models.deep_learning import SelectiveMobileNet

# Load
predictor = SelectiveMobileNet(
    num_classes=10,
    confidence_threshold=0.75,
    temperature=meta["temperature"],
    device="cuda",
)
predictor.load("models/checkpoints/mobilenet_best.pth")

# Preprocess
tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Sample 5 random test images
test_root = Path("data/processed/test")
samples = []
for cls_dir in test_root.iterdir():
    if cls_dir.is_dir():
        imgs = list(cls_dir.glob("*.jpg"))
        if imgs:
            samples.append((cls_dir.name, random.choice(imgs)))
samples = random.sample(samples, min(5, len(samples)))

class_names = meta["class_names"]
for true_class, img_path in samples:
    x = tf(Image.open(img_path).convert("RGB")).unsqueeze(0)
    pred = predictor.predict_with_abstention(x)
    if pred.abstained:
        label = f"ABSTAINED (conf={pred.confidence:.3f})"
    else:
        label = f"{class_names[pred.class_idx]:<12s} (conf={pred.confidence:.3f})"
    mark = "✓" if (not pred.abstained and class_names[pred.class_idx] == true_class) else "✗"
    print(f"  {mark} true={true_class:<12s} pred={label}")


## ✅ Done — next steps

**Download from Drive to your Mac**:
   - `mobilenet_best.pth` → `models/checkpoints/`
   - `mobilenet_meta.json` → `models/checkpoints/`




In [ ]:
from google.colab import files
files.download("models/checkpoints/mobilenet_best.pth")
files.download("models/checkpoints/mobilenet_meta.json")